# Hackapizza 2.0 — Batch Ingestion su Kaggle GPU

**Obiettivo**: Parsare tutti i documenti del dataset, generare embedding con GPU Kaggle (T4),
esportare un file `corpus.json` con chunks + vettori pronti da usare in locale.

**Come usare**:
1. Caricare questo notebook su Kaggle
2. Abilitare GPU: Settings → Accelerator → GPU T4 x2
3. Caricare il dataset come Kaggle Dataset
4. Eseguire tutte le celle
5. Scaricare `corpus.json` dall'output

In [ ]:
# Installa dipendenze
!pip install sentence-transformers rank-bm25 docling einops -q

In [ ]:
import os, json, time
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Carica modello embedding
print('Caricamento modello...')
t0 = time.time()
model = SentenceTransformer('nomic-ai/nomic-embed-text-v1', trust_remote_code=True)
model = model.to(DEVICE)
print(f'Pronto in {time.time()-t0:.1f}s')

In [ ]:
# ⚠️ Aggiorna questo path con il path del dataset su Kaggle
DATA_DIR = Path('/kaggle/input/hackapizza-dataset/')  # modifica se necessario

files = list(DATA_DIR.rglob('*'))
files = [f for f in files if f.is_file()]
print(f'Trovati {len(files)} file')
for f in files:
    print(f'  {f.suffix:6} {f.name}')

In [ ]:
def parse_document(file_path: Path) -> str:
    suffix = file_path.suffix.lower()
    try:
        if suffix in ['.pdf', '.docx']:
            from docling.document_converter import DocumentConverter
            result = DocumentConverter().convert(str(file_path))
            return result.document.export_to_markdown()
        elif suffix in ['.html', '.htm']:
            from bs4 import BeautifulSoup
            with open(file_path, encoding='utf-8', errors='replace') as f:
                return BeautifulSoup(f.read(), 'html.parser').get_text(separator='\n', strip=True)
        elif suffix == '.csv':
            import pandas as pd
            return pd.read_csv(file_path, on_bad_lines='skip').to_string()
        else:
            with open(file_path, encoding='utf-8', errors='replace') as f:
                return f.read()
    except Exception as e:
        print(f'  ERRORE parsing {file_path.name}: {e}')
        return ''


def chunk_text(text: str, chunk_size: int = 512, overlap: int = 64) -> list:
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

In [ ]:
# Parsing + chunking di tutti i documenti
all_chunks = []

for file_path in tqdm(files, desc='Parsing'):
    text = parse_document(file_path)
    if not text.strip():
        continue
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text': chunk,
            'source': file_path.name,
            'chunk_index': i,
        })

print(f'\nTotale chunk: {len(all_chunks)}')

In [ ]:
# Generazione embedding in batch (veloce su GPU)
texts = [c['text'] for c in all_chunks]

print(f'Embedding {len(texts)} chunk su {DEVICE}...')
t0 = time.time()
vectors = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)
print(f'Completato in {time.time()-t0:.1f}s  |  Shape: {vectors.shape}')

# Aggiungi vettori ai chunk
for chunk, vec in zip(all_chunks, vectors):
    chunk['vector'] = vec.tolist()

In [ ]:
# Salva corpus completo
output_path = '/kaggle/working/corpus.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, ensure_ascii=False)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f'Salvato: {output_path} ({size_mb:.1f} MB)')
print(f'Chunk totali: {len(all_chunks)}')
print(f'Dimensione vettori: {len(all_chunks[0]["vector"])}')
print('\nFatto! Scarica corpus.json dal pannello Output.')